# Gold Layer - Dimension: Customers

This notebook builds the Gold customer dimension table by combining curated Silver customer, ERP profile, and location data.

## Tools and Operations
- **Spark SQL + PySpark** for joins and final projections
- Data quality checks (duplicate and null checks)
- Write to Gold layer as a managed table

## Output
- `workspace.gold.customers`


## 1) Inspect Input Tables


In [0]:
%sql
USE CATALOG workspace;

SELECT * FROM silver.crm_customers LIMIT 5

In [0]:
%sql
SELECT * FROM silver.erp_customers LIMIT 5

In [0]:
%sql
SELECT * FROM silver.erp_customer_location  LIMIT 5

## 2) Transformations


### Setup


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

### Join Tables


In [0]:
%python

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA silver")

query = """
SELECT
  cc.*,
  ec.birth_date,
  el.country
FROM crm_customers cc
LEFT JOIN erp_customers ec ON cc.customer_key = ec.customer_key
LEFT JOIN erp_customer_location el ON cc.customer_key = el.customer_key
"""
df = spark.sql(query)

display(df)

In [0]:
df.printSchema()

### Duplicate Checks


In [0]:
print(df.count())
print(df.select("customer_id").distinct().count())
print(df.select("customer_key").distinct().count())

### Null Checks


In [0]:

df.select([
    F.sum(F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))).alias(c)
    for c in df.columns
]).display()

### Build Final Projection


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA silver")

query = """
SELECT
  CAST(cc.customer_id AS INT) AS customer_id,
  cc.customer_key,
  COALESCE(cc.first_name, "Unknown") AS first_name,
  COALESCE(cc.last_name, "Unknown") AS last_name,
  COALESCE(cc.gender, "Unknown") AS gender,
  COALESCE(cc.marital_status, "Unknown") AS marital_status,
  COALESCE(el.country, "Unknown") AS country,
  ec.birth_date,
  cc.create_date
FROM crm_customers cc
LEFT JOIN erp_customers ec ON cc.customer_key = ec.customer_key
LEFT JOIN erp_customer_location el ON cc.customer_key = el.customer_key
"""

df = spark.sql(query)

df.select([
    F.sum(F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))).alias(c)
    for c in df.columns
]).display()

In [0]:
df.display()
print(df.select("customer_key").distinct().count())

## 3) Write Gold Table


In [0]:
df.write.mode("overwrite").saveAsTable("workspace.gold.customers")

## 4) Validation


In [0]:
%sql
SELECT * FROM workspace.gold.customers LIMIT 10